In [ ]:
!pip install docling beautifulsoup4 requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 597.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.4/370.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.4/240.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.0 MB/s eta 0:00:00
   ━━━

In [ ]:
!pip install -q docling beautifulsoup4 requests pypdf

import re
import json
import time
import hashlib
import requests
from pathlib import Path
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from docling.document_converter import DocumentConverter

try:
    from pypdf import PdfReader
except Exception:
    PdfReader = None

_CONVERTER = None
CACHE_DIR = Path(".pipeline_cache")
CACHE_DIR.mkdir(exist_ok=True)

def _get_converter():
    global _CONVERTER
    if _CONVERTER is None:
        _CONVERTER = DocumentConverter()
    return _CONVERTER

def _is_pdf_source(source: str) -> bool:
    return urlparse(source).path.lower().endswith(".pdf") if source.startswith("http") else source.lower().endswith(".pdf")

def _cache_key_for_source(source: str) -> str:
    if source.startswith("http"):
        return hashlib.sha256(source.encode("utf-8")).hexdigest()
    p = Path(source)
    try:
        st = p.stat()
        sig = f"{p.resolve()}::{st.st_size}::{int(st.st_mtime)}"
    except Exception:
        sig = str(p)
    return hashlib.sha256(sig.encode("utf-8")).hexdigest()

def _download_pdf_to_cache(url: str) -> Path:
    key = hashlib.sha256(url.encode("utf-8")).hexdigest()
    pdf_path = CACHE_DIR / f"{key}.pdf"
    if pdf_path.exists() and pdf_path.stat().st_size > 0:
        return pdf_path
    with requests.get(url, timeout=90, stream=True) as r:
        r.raise_for_status()
        with open(pdf_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 256):
                if chunk:
                    f.write(chunk)
    return pdf_path

def extract_pdf_raw_txt(source, out_path="raw_extracted.txt", use_cache=True, prefer_fast=True):
    source_key = _cache_key_for_source(source)
    raw_cache = CACHE_DIR / f"{source_key}.raw.txt"
    if use_cache and raw_cache.exists() and raw_cache.stat().st_size > 0:
        raw = raw_cache.read_text(encoding="utf-8", errors="ignore")
        Path(out_path).write_text(raw, encoding="utf-8")
        return raw

    pdf_path = _download_pdf_to_cache(source) if source.startswith("http") else Path(source)
    raw = ""

    if prefer_fast and PdfReader is not None:
        try:
            t0 = time.time()
            reader = PdfReader(str(pdf_path))
            fast = "\n\n".join((p.extract_text() or "") for p in reader.pages).strip()
            if len(fast) > 1500:
                raw = fast
                print(f"[extract_pdf_raw_txt] fast path: {time.time()-t0:.2f}s")
        except Exception:
            raw = ""

    if not raw:
        t0 = time.time()
        conv = _get_converter()
        raw = conv.convert(str(pdf_path)).document.export_to_markdown().strip()
        print(f"[extract_pdf_raw_txt] docling path: {time.time()-t0:.2f}s")

    Path(out_path).write_text(raw, encoding="utf-8")
    if use_cache:
        raw_cache.write_text(raw, encoding="utf-8")
    return raw

def extract_html_raw_txt(url, out_path="raw_extracted.txt"):
    html = requests.get(url, timeout=30).text
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "header", "footer", "aside"]):
        tag.decompose()
    text = "\n".join(line.strip() for line in soup.get_text("\n").splitlines() if line.strip())
    Path(out_path).write_text(text, encoding="utf-8")
    return text

def clean_text(raw_text, out_path="cleaned.txt"):
    text = raw_text
    text = re.sub(r"<!--\s*image\s*-->", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*\|[-\s|:]+\|\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"https?\s*:\s*/\s*/", lambda m: m.group(0).replace(" ", ""), text)

    lines = []
    for line in text.splitlines():
        s = re.sub(r"[ \t]+", " ", line.strip())
        if not s:
            lines.append("")
            continue
        if s.startswith("|") and s.endswith("|"):
            continue
        if re.fullmatch(r"Page\s*\d+", s, flags=re.IGNORECASE):
            continue
        lines.append(s)

    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    Path(out_path).write_text(text, encoding="utf-8")
    return text

def extract_topics_from_cleaned(text):
    lines = [ln.strip() for ln in text.splitlines()]

    def is_markdown_heading(s):
        return bool(re.match(r"^#{1,6}\s+\S", s))

    def strip_markdown_heading(s):
        return re.sub(r"^#{1,6}\s*", "", s).strip()

    def is_roman_or_numbered(s):
        return bool(re.match(r"^(?:[IVXLC]+\.|\d+\.)\s+\S", s, flags=re.IGNORECASE))

    def is_caps_heading(s):
        if len(s) < 4 or len(s) > 180:
            return False
        if s.endswith("."):
            return False
        letters = [c for c in s if c.isalpha()]
        if len(letters) < 4:
            return False
        return (sum(c.isupper() for c in letters) / len(letters)) > 0.85

    def looks_like_toc_line(s):
        return bool(re.search(r"\.{2,}\s*\d+\s*$", s))

    def is_heading_line(s):
        if not s:
            return False
        return is_markdown_heading(s) or is_roman_or_numbered(s) or is_caps_heading(s)

    topics = []
    current_title = "Document"
    body = []
    seen_real_body = False
    i = 0
    n = len(lines)

    def flush():
        nonlocal body, current_title, seen_real_body
        txt = "\n".join(x for x in body if x).strip()
        if txt:
            topics.append({"topic": current_title, "text": txt})
            seen_real_body = True
        body = []

    while i < n:
        s = lines[i]

        if not s:
            if body and body[-1] != "":
                body.append("")
            i += 1
            continue

        # Remove TOC-style lines only before first body topic
        if not seen_real_body and looks_like_toc_line(s):
            i += 1
            continue

        if is_heading_line(s):
            heading_parts = []
            while i < n and lines[i] and is_heading_line(lines[i]):
                part = strip_markdown_heading(lines[i]) if is_markdown_heading(lines[i]) else lines[i]
                heading_parts.append(part)
                i += 1
            heading = re.sub(r"\s+", " ", " ".join(heading_parts)).strip()
            flush()
            current_title = heading if heading else "Document"
            continue

        body.append(s)
        i += 1

    flush()
    return topics

def merge_and_rechunk_by_topic(chunks, max_words=500, overlap_ratio=0.2):
    from collections import OrderedDict

    def words(s):
        return re.findall(r"\S+", s or "")

    merged = OrderedDict()
    for c in chunks:
        topic = (c.get("topic") or "Document").strip()
        txt = re.sub(r"\s+", " ", (c.get("text") or "")).strip()
        if not txt:
            continue
        merged[topic] = (merged.get(topic, "") + " " + txt).strip()

    overlap_words = max(1, int(max_words * overlap_ratio))
    sent_re = re.compile(r"(?<=[.!?])\s+")
    out = []

    for topic, full in merged.items():
        sents = [s.strip() for s in sent_re.split(full) if s.strip()]
        cur = []

        def flush():
            if cur:
                out.append({"topic": topic, "text": " ".join(cur)})

        for s in sents:
            sw = words(s)
            if not sw:
                continue
            if len(sw) > max_words:
                flush()
                cur = []
                start = 0
                while start < len(sw):
                    end = min(start + max_words, len(sw))
                    piece = sw[start:end]
                    out.append({"topic": topic, "text": " ".join(piece)})
                    if end == len(sw):
                        cur = sw[max(0, end - overlap_words):end]
                        break
                    start = end - overlap_words
                continue

            if len(cur) + len(sw) <= max_words:
                cur.extend(sw)
            else:
                flush()
                cur = (cur[-overlap_words:] if cur else []) + sw

        flush()

    return out

def save_all(topics, chunks, topics_path="topics.txt", chunks_path="chunks.json"):
    with open(topics_path, "w", encoding="utf-8") as f:
        for i, t in enumerate(topics, 1):
            f.write(f"\n--- TOPIC {i} ---\n[{t.get('topic','Document')}]\n{t.get('text','')}\n")
    with open(chunks_path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)

def process_document(source, prefer_fast_pdf=True, use_cache=True):
    raw = extract_pdf_raw_txt(source, use_cache=use_cache, prefer_fast=prefer_fast_pdf) if _is_pdf_source(source) else extract_html_raw_txt(source)
    cleaned = clean_text(raw)
    topics = extract_topics_from_cleaned(cleaned)
    chunks = merge_and_rechunk_by_topic(topics, max_words=500, overlap_ratio=0.2)
    save_all(topics, chunks)
    return topics, chunks


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.2/331.2 kB 5.9 MB/s eta 0:00:00


In [ ]:
pdf_url = "https://www.iit.edu/sites/default/files/2025-08/25-26_StudentHandbook_Final_Aug15.pdf"
topics, chunks = process_document(pdf_url)
print("Topics:", len(topics))
print("Chunks:", len(chunks))

Topics: 281
Chunks: 340


In [ ]:
# Word count per chunk
for i, c in enumerate(chunks, 1):
    wc = len(re.findall(r"\S+", c["text"]))
    print(f"Chunk {i}: {wc} words | topic: {c['topic']}")

Chunk 1: 8 words | topic: Document
Chunk 2: 154 words | topic: CONTENTS
Chunk 3: 261 words | topic: WELCOME TO ILLINOIS INSTITUTE OF TECHNOLOGY AND THE 2025-2026 ACADEMIC YEAR!
Chunk 4: 393 words | topic: A BRIEF HISTORY OF ILLINOIS INSTITUTE OF TECHNOLOGY
Chunk 5: 21 words | topic: ILLINOIS TECH’S MISSION
Chunk 6: 38 words | topic: ILLINOIS TECH’S VISION
Chunk 7: 67 words | topic: ILLINOIS TECH’S COMMITMENT TO DIVERSITY AND INCLUSION
Chunk 8: 75 words | topic: TELEPHONE DIALING DIRECTIONS
Chunk 9: 69 words | topic: ILLINOIS TECH CAMPUS ADDRESSES
Chunk 10: 158 words | topic: MIES CAMPUS BUILDINGS
Chunk 11: 97 words | topic: STUDENT HOUSING
Chunk 12: 158 words | topic: ILLINOIS TECH MIES CAMPUS ACADEMIC DEPARTMENTS AND LOCATIONS
Chunk 13: 62 words | topic: VOTER REGISTRATION INFORMATION
Chunk 14: 146 words | topic: STUDENT ORGANIZATIONS AND ACTIVITIES
Chunk 15: 270 words | topic: OTHER LEADERSHIP OPPORTUNITIES
Chunk 16: 260 words | topic: FRATERNITIES AND SORORITIES
Chunk 17: 117 words 